In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

current = Path.cwd()

project_root = None

for candidate in [current, *current.parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(
        "Could not find CODEZILLA project root."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)

Project root: e:\CODEZILLA-SIH26145


In [2]:
ENCRYPTED_FILE = (
    project_root
    / "data"
    / "raw"
    / "CTU-13"
    / "capture20110810.binetflow"
)

encrypted_df = pd.read_csv(
    ENCRYPTED_FILE,
    usecols=[
        "StartTime",
        "Dur",
        "Proto",
        "SrcAddr",
        "Sport",
        "DstAddr",
        "Dport",
        "TotPkts",
        "TotBytes",
        "SrcBytes",
        "Label"
    ]
)

encrypted_df["StartTime"] = pd.to_datetime(
    encrypted_df["StartTime"],
    errors="coerce"
)

encrypted_df["Dport_numeric"] = pd.to_numeric(
    encrypted_df["Dport"]
        .astype(str)
        .str.strip(),
    errors="coerce"
)

encrypted_df = encrypted_df.dropna(
    subset=[
        "StartTime",
        "SrcAddr",
        "DstAddr"
    ]
)

print(
    "Encrypted analysis dataframe:",
    encrypted_df.shape
)

Encrypted analysis dataframe: (2824636, 12)


In [3]:
encrypted_ports = [
    443,
    8443,
    465,
    993,
    995
]

encrypted_flows = encrypted_df[
    encrypted_df["Dport_numeric"].isin(
        encrypted_ports
    )
].copy()

print(
    "Candidate encrypted flows:",
    len(encrypted_flows)
)

print("\nProtocol distribution:")
print(
    encrypted_flows["Proto"]
    .value_counts()
)

print("\nDestination port distribution:")
print(
    encrypted_flows["Dport_numeric"]
    .value_counts()
)

Candidate encrypted flows: 74661

Protocol distribution:
Proto
tcp    52995
udp    21666
Name: count, dtype: int64

Destination port distribution:
Dport_numeric
443.0     70965
993.0      2651
995.0       999
465.0        44
8443.0        2
Name: count, dtype: int64


In [4]:
print("\nLabels associated with candidate encrypted traffic:")

display(
    encrypted_flows["Label"]
    .value_counts()
    .head(40)
)



Labels associated with candidate encrypted traffic:


Label
flow=Background-TCP-Established                            31546
flow=Background-UDP-Established                            21096
flow=Background-Established-cmpgw-CVUT                     15271
flow=Background                                             1163
flow=From-Normal-V42-Stribrek                               1030
flow=Background-TCP-Attempt                                  924
flow=Background-google-webmail                               534
flow=From-Normal-V42-Jist                                    417
flow=Background-Attempt-cmpgw-CVUT                           314
flow=Background-google-pop                                   188
flow=Background-google-analytics6                            143
flow=Background-google-analytics13                           141
flow=Background-google-analytics11                           138
flow=Background-UDP-Attempt                                  121
flow=From-Normal-V42-Grill                                   116
flow=Background-goo

In [5]:
# ============================================================
# STEP 213 — ENCRYPTED THREAT GROUND TRUTH
# ============================================================

encrypted_flows["encrypted_threat"] = (
    encrypted_flows["Label"]
    .astype(str)
    .str.contains(
        r"From-Botnet.*(SSL|Encryption)",
        case=False,
        regex=True
    )
    .astype(int)
)

print("Encrypted target distribution:")
print(
    encrypted_flows["encrypted_threat"]
    .value_counts()
)

print("\nEncrypted threat labels:")
display(
    encrypted_flows.loc[
        encrypted_flows["encrypted_threat"] == 1,
        "Label"
    ].value_counts()
)

Encrypted target distribution:
encrypted_threat
0    74587
1       74
Name: count, dtype: int64

Encrypted threat labels:


C:\Users\magad\AppData\Local\Temp\ipykernel_30564\2076105532.py:8: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


Label
flow=From-Botnet-V42-TCP-WEB-Established-SSL               50
flow=From-Botnet-V42-TCP-Established-SSL-To-Microsoft-4    13
flow=From-Botnet-V42-TCP-CC54-Custom-Encryption            11
Name: count, dtype: int64

In [6]:
# ============================================================
# STEP 214 — SOURCE-CENTRIC ENCRYPTED WINDOWS
# ============================================================

encrypted_flows = encrypted_flows.sort_values(
    ["SrcAddr", "StartTime"]
).copy()

encrypted_flows["time_window"] = (
    encrypted_flows["StartTime"]
    .dt.floor("10s")
)

encrypted_window = (
    encrypted_flows
    .groupby(
        ["SrcAddr", "time_window"]
    )
    .agg(
        encrypted_flow_count=(
            "StartTime",
            "size"
        ),

        encrypted_total_packets=(
            "TotPkts",
            "sum"
        ),

        encrypted_total_bytes=(
            "TotBytes",
            "sum"
        ),

        encrypted_unique_destinations=(
            "DstAddr",
            "nunique"
        ),

        encrypted_unique_ports=(
            "Dport_numeric",
            "nunique"
        ),

        encrypted_mean_duration=(
            "Dur",
            "mean"
        ),

        encrypted_target=(
            "encrypted_threat",
            "max"
        )
    )
    .reset_index()
)

print(
    "Encrypted source-centric windows:",
    encrypted_window.shape
)

print("\nTarget distribution:")
print(
    encrypted_window[
        "encrypted_target"
    ].value_counts()
)

display(
    encrypted_window.head(20)
)

Encrypted source-centric windows: (47507, 9)

Target distribution:
encrypted_target
0    47433
1       74
Name: count, dtype: int64


,SrcAddr,time_window,encrypted_flow_count,encrypted_total_packets,encrypted_total_bytes,encrypted_unique_destinations,encrypted_unique_ports,encrypted_mean_duration,encrypted_target
0,1.108.169.143,2011-08-10 14:50:00,1,12,756,1,1,64.778000,0
1,1.113.23.252,2011-08-10 13:04:20,1,13,1084,1,1,1.339552,0
2,1.114.175.135,2011-08-10 14:06:20,1,21,2320,1,1,83.150185,0
3,1.126.227.134,2011-08-10 14:29:10,1,2,128,1,1,0.000826,0
4,1.127.173.21,2011-08-10 12:44:50,1,9,572,1,1,218.938843,0
5,1.127.173.21,2011-08-10 13:02:30,1,9,572,1,1,213.635666,0
6,1.127.173.21,2011-08-10 13:13:10,1,5,308,1,1,214.103119,0
7,1.127.173.21,2011-08-10 13:26:20,1,5,324,1,1,192.901199,0
8,1.127.173.21,2011-08-10 13:40:30,1,9,572,1,1,209.456589,0
9,1.127.173.21,2011-08-10 13:54:20,1,9,572,1,1,208.594803,0


In [7]:
# ============================================================
# STEP 215 — ENCRYPTED BEHAVIOURAL FEATURES
# ============================================================

encrypted_window = (
    encrypted_window
    .sort_values(
        ["SrcAddr", "time_window"]
    )
    .reset_index(drop=True)
)

group = encrypted_window.groupby(
    "SrcAddr",
    group_keys=False
)

encrypted_window["bytes_per_flow"] = (
    encrypted_window["encrypted_total_bytes"]
    /
    encrypted_window["encrypted_flow_count"]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

encrypted_window["packets_per_flow"] = (
    encrypted_window["encrypted_total_packets"]
    /
    encrypted_window["encrypted_flow_count"]
).replace(
    [np.inf, -np.inf],
    np.nan
).fillna(0)

encrypted_window["flow_count_prev"] = (
    group["encrypted_flow_count"]
    .shift(1)
)

encrypted_window["flow_count_change"] = (
    encrypted_window["encrypted_flow_count"]
    -
    encrypted_window["flow_count_prev"]
).fillna(0)

encrypted_window["bytes_prev"] = (
    group["encrypted_total_bytes"]
    .shift(1)
)

encrypted_window["bytes_change"] = (
    encrypted_window["encrypted_total_bytes"]
    -
    encrypted_window["bytes_prev"]
).fillna(0)

encrypted_window["destination_change"] = (
    encrypted_window["encrypted_unique_destinations"]
    -
    group["encrypted_unique_destinations"]
    .shift(1)
).fillna(0)

encrypted_features = [
    "encrypted_flow_count",
    "encrypted_total_packets",
    "encrypted_total_bytes",
    "encrypted_unique_destinations",
    "encrypted_unique_ports",
    "encrypted_mean_duration",
    "bytes_per_flow",
    "packets_per_flow",
    "flow_count_change",
    "bytes_change",
    "destination_change"
]

encrypted_window[encrypted_features] = (
    encrypted_window[encrypted_features]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

print(
    "Encrypted feature count:",
    len(encrypted_features)
)

display(
    encrypted_window[
        [
            "SrcAddr",
            "time_window"
        ] + encrypted_features + [
            "encrypted_target"
        ]
    ].head(20)
)

Encrypted feature count: 11


,SrcAddr,time_window,encrypted_flow_count,encrypted_total_packets,encrypted_total_bytes,encrypted_unique_destinations,encrypted_unique_ports,encrypted_mean_duration,bytes_per_flow,packets_per_flow,flow_count_change,bytes_change,destination_change,encrypted_target
0,1.108.169.143,2011-08-10 14:50:00,1,12,756,1,1,64.778000,756.0,12.0,0.0,0.0,0.0,0
1,1.113.23.252,2011-08-10 13:04:20,1,13,1084,1,1,1.339552,1084.0,13.0,0.0,0.0,0.0,0
2,1.114.175.135,2011-08-10 14:06:20,1,21,2320,1,1,83.150185,2320.0,21.0,0.0,0.0,0.0,0
3,1.126.227.134,2011-08-10 14:29:10,1,2,128,1,1,0.000826,128.0,2.0,0.0,0.0,0.0,0
4,1.127.173.21,2011-08-10 12:44:50,1,9,572,1,1,218.938843,572.0,9.0,0.0,0.0,0.0,0
5,1.127.173.21,2011-08-10 13:02:30,1,9,572,1,1,213.635666,572.0,9.0,0.0,0.0,0.0,0
6,1.127.173.21,2011-08-10 13:13:10,1,5,308,1,1,214.103119,308.0,5.0,0.0,-264.0,0.0,0
7,1.127.173.21,2011-08-10 13:26:20,1,5,324,1,1,192.901199,324.0,5.0,0.0,16.0,0.0,0
8,1.127.173.21,2011-08-10 13:40:30,1,9,572,1,1,209.456589,572.0,9.0,0.0,248.0,0.0,0
9,1.127.173.21,2011-08-10 13:54:20,1,9,572,1,1,208.594803,572.0,9.0,0.0,0.0,0.0,0


In [8]:
# ============================================================
# STEP 216 — CLASS SEPARATION
# ============================================================

display(
    encrypted_window
    .groupby("encrypted_target")[
        encrypted_features
    ]
    .mean()
    .T
)

encrypted_target,0,1
encrypted_flow_count,1.572365,1.067568
encrypted_total_packets,138.424346,103.932432
encrypted_total_bytes,83630.504079,85618.662162
encrypted_unique_destinations,1.235996,1.067568
encrypted_unique_ports,1.019586,1.000000
encrypted_mean_duration,145.753556,19.743423
bytes_per_flow,25960.813942,85561.527027
packets_per_flow,65.737616,103.317568
flow_count_change,-0.000632,0.067568
bytes_change,914.534691,2437.783784


In [9]:
# ============================================================
# STEP 217 — ENCRYPTED THREAT TEMPORAL DISTRIBUTION
# ============================================================

positive_windows = (
    encrypted_window[
        encrypted_window["encrypted_target"] == 1
    ]
    .sort_values("time_window")
    .copy()
)

print(
    "Total encrypted-threat windows:",
    len(positive_windows)
)

print("\nFirst threat window:")
print(
    positive_windows["time_window"].min()
)

print("\nLast threat window:")
print(
    positive_windows["time_window"].max()
)

print("\nThreat windows by rough time quartile:")

display(
    positive_windows[
        ["time_window", "SrcAddr", "encrypted_target"]
    ].head(30)
)

Total encrypted-threat windows: 74

First threat window:
2011-08-10 11:07:00

Last threat window:
2011-08-10 15:21:50

Threat windows by rough time quartile:


,time_window,SrcAddr,encrypted_target
10163,2011-08-10 11:07:00,147.32.84.165,1
10167,2011-08-10 11:08:50,147.32.84.165,1
10169,2011-08-10 11:10:10,147.32.84.165,1
10176,2011-08-10 11:12:50,147.32.84.165,1
10178,2011-08-10 11:13:30,147.32.84.165,1
10180,2011-08-10 11:14:10,147.32.84.165,1
10182,2011-08-10 11:15:10,147.32.84.165,1
10185,2011-08-10 11:16:20,147.32.84.165,1
10191,2011-08-10 11:18:50,147.32.84.165,1
10201,2011-08-10 11:22:30,147.32.84.165,1


In [10]:
# ============================================================
# STEP 218 — ENCRYPTED TEMPORAL SPLIT
# ============================================================

encrypted_window = (
    encrypted_window
    .sort_values("time_window")
    .reset_index(drop=True)
)

n = len(encrypted_window)

train_end = int(n * 0.60)
val_end = int(n * 0.80)

encrypted_train = (
    encrypted_window
    .iloc[:train_end]
    .copy()
)

encrypted_val = (
    encrypted_window
    .iloc[train_end:val_end]
    .copy()
)

encrypted_test = (
    encrypted_window
    .iloc[val_end:]
    .copy()
)

print("TRAIN :", encrypted_train.shape)
print("VAL   :", encrypted_val.shape)
print("TEST  :", encrypted_test.shape)

print("\nTrain target:")
print(
    encrypted_train["encrypted_target"]
    .value_counts()
)

print("\nValidation target:")
print(
    encrypted_val["encrypted_target"]
    .value_counts()
)

print("\nTest target:")
print(
    encrypted_test["encrypted_target"]
    .value_counts()
)

TRAIN : (28504, 16)
VAL   : (9501, 16)
TEST  : (9502, 16)

Train target:
encrypted_target
0    28457
1       47
Name: count, dtype: int64

Validation target:
encrypted_target
0    9497
1       4
Name: count, dtype: int64

Test target:
encrypted_target
0    9479
1      23
Name: count, dtype: int64


In [11]:
# ============================================================
# STEP 218A — INSPECT POSITIVE COUNTS BY TIME
# ============================================================

temp = (
    encrypted_window
    .sort_values("time_window")
    .reset_index(drop=True)
    .copy()
)

# Divide the timeline into 10 equal chronological blocks.
temp["time_block"] = pd.qcut(
    temp.index,
    q=10,
    labels=False,
    duplicates="drop"
)

block_summary = (
    temp
    .groupby("time_block")
    .agg(
        windows=("encrypted_target", "size"),
        threats=("encrypted_target", "sum"),
        first_time=("time_window", "min"),
        last_time=("time_window", "max")
    )
    .reset_index()
)

display(block_summary)

,time_block,windows,threats,first_time,last_time
0,0,4751,0,2011-08-10 09:46:50,2011-08-10 10:25:20
1,1,4751,0,2011-08-10 10:25:20,2011-08-10 11:03:30
2,2,4750,28,2011-08-10 11:03:30,2011-08-10 11:42:20
3,3,4751,8,2011-08-10 11:42:20,2011-08-10 12:22:00
4,4,4751,2,2011-08-10 12:22:00,2011-08-10 13:00:30
5,5,4750,9,2011-08-10 13:00:30,2011-08-10 13:37:00
6,6,4751,4,2011-08-10 13:37:00,2011-08-10 14:12:20
7,7,4750,0,2011-08-10 14:12:20,2011-08-10 14:48:10
8,8,4751,20,2011-08-10 14:48:10,2011-08-10 15:21:00
9,9,4751,3,2011-08-10 15:21:00,2011-08-10 15:54:00


In [12]:
# ============================================================
# STEP 218B — CORRECT TEMPORAL SPLIT
# ============================================================

encrypted_data = (
    encrypted_window
    .sort_values("time_window")
    .reset_index(drop=True)
    .copy()
)

# Divide into the same 10 chronological blocks we inspected.
encrypted_data["time_block"] = pd.qcut(
    encrypted_data.index,
    q=10,
    labels=False,
    duplicates="drop"
)

# ------------------------------------------------------------
# TRAIN = blocks 0-4
# VALIDATION = blocks 5-6
# TEST = blocks 7-9
# ------------------------------------------------------------

encrypted_train = encrypted_data[
    encrypted_data["time_block"].between(0, 4)
].copy()

encrypted_val = encrypted_data[
    encrypted_data["time_block"].between(5, 6)
].copy()

encrypted_test = encrypted_data[
    encrypted_data["time_block"].between(7, 9)
].copy()

print("TRAIN:", encrypted_train.shape)
print("VALIDATION:", encrypted_val.shape)
print("TEST:", encrypted_test.shape)

print("\nTrain target:")
print(
    encrypted_train["encrypted_target"]
    .value_counts()
)

print("\nValidation target:")
print(
    encrypted_val["encrypted_target"]
    .value_counts()
)

print("\nTest target:")
print(
    encrypted_test["encrypted_target"]
    .value_counts()
)

print("\nTime ranges:")

print(
    "Train:",
    encrypted_train["time_window"].min(),
    "→",
    encrypted_train["time_window"].max()
)

print(
    "Validation:",
    encrypted_val["time_window"].min(),
    "→",
    encrypted_val["time_window"].max()
)

print(
    "Test:",
    encrypted_test["time_window"].min(),
    "→",
    encrypted_test["time_window"].max()
)

TRAIN: (23754, 17)
VALIDATION: (9501, 17)
TEST: (14252, 17)

Train target:
encrypted_target
0    23716
1       38
Name: count, dtype: int64

Validation target:
encrypted_target
0    9488
1      13
Name: count, dtype: int64

Test target:
encrypted_target
0    14229
1       23
Name: count, dtype: int64

Time ranges:
Train: 2011-08-10 09:46:50 → 2011-08-10 13:00:30
Validation: 2011-08-10 13:00:30 → 2011-08-10 14:12:20
Test: 2011-08-10 14:12:20 → 2011-08-10 15:54:00


In [13]:
# ============================================================
# STEP 219B — PREPARE ENCRYPTED MODEL DATA
# ============================================================

encrypted_features = [
    "encrypted_flow_count",
    "encrypted_total_packets",
    "encrypted_total_bytes",
    "encrypted_unique_destinations",
    "encrypted_unique_ports",
    "encrypted_mean_duration",
    "bytes_per_flow",
    "packets_per_flow",
    "flow_count_change",
    "bytes_change",
    "destination_change"
]

X_enc_train = (
    encrypted_train[encrypted_features]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

y_enc_train = encrypted_train[
    "encrypted_target"
].astype(int)

X_enc_val = (
    encrypted_val[encrypted_features]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

y_enc_val = encrypted_val[
    "encrypted_target"
].astype(int)

X_enc_test = (
    encrypted_test[encrypted_features]
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)

y_enc_test = encrypted_test[
    "encrypted_target"
].astype(int)

print("Feature count:", len(encrypted_features))

Feature count: 11


In [14]:
# ============================================================
# STEP 220B — TRAIN ENCRYPTED HGB
# ============================================================

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight

encrypted_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_enc_train
)

encrypted_hgb = HistGradientBoostingClassifier(
    max_iter=350,
    learning_rate=0.04,
    max_leaf_nodes=15,
    min_samples_leaf=5,
    l2_regularization=2.0,
    random_state=42
)

encrypted_hgb.fit(
    X_enc_train,
    y_enc_train,
    sample_weight=encrypted_weights
)

print("✅ Encrypted HGB trained")

c:\Users\magad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\magad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

✅ Encrypted HGB trained


In [15]:
# ============================================================
# STEP 221B — VALIDATION PROBABILITIES
# ============================================================

enc_val_proba = (
    encrypted_hgb
    .predict_proba(X_enc_val)[:, 1]
)

print(
    pd.Series(enc_val_proba).describe()
)

print("\nMean score by class:")

display(
    pd.DataFrame({
        "target": y_enc_val.to_numpy(),
        "score": enc_val_proba
    })
    .groupby("target")["score"]
    .describe()
)

count    9501.000000
mean        0.001699
std         0.028396
min         0.000004
25%         0.000036
50%         0.000036
75%         0.000065
max         0.999021
dtype: float64

Mean score by class:


,count,mean,std,min,25%,50%,75%,max
target,,,,,,,,
0,9488.0,0.001487,0.024449,0.000004,0.000036,0.000036,0.000065,0.994566
1,13.0,0.156499,0.373905,0.000008,0.000015,0.001047,0.015794,0.999021


In [17]:
# ============================================================
# FIX — ENCRYPTED MODEL EVALUATION IMPORTS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

print("✅ Encrypted evaluation metrics imported")

✅ Encrypted evaluation metrics imported


In [18]:
# ============================================================
# STEP 222B — ENCRYPTED THRESHOLD SEARCH
# ============================================================

encrypted_threshold_rows = []

for threshold in np.arange(
    0.05,
    0.96,
    0.01
):

    pred = (
        enc_val_proba >= threshold
    ).astype(int)

    encrypted_threshold_rows.append({
        "threshold": round(
            float(threshold),
            2
        ),

        "precision": precision_score(
            y_enc_val,
            pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_enc_val,
            pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_enc_val,
            pred,
            zero_division=0
        )
    })

encrypted_threshold_table = pd.DataFrame(
    encrypted_threshold_rows
)

display(
    encrypted_threshold_table
    .sort_values(
        "f1",
        ascending=False
    )
    .head(20)
)

,threshold,precision,recall,f1
90,0.95,0.500000,0.153846,0.235294
89,0.94,0.500000,0.153846,0.235294
80,0.85,0.400000,0.153846,0.222222
72,0.77,0.400000,0.153846,0.222222
73,0.78,0.400000,0.153846,0.222222
74,0.79,0.400000,0.153846,0.222222
75,0.80,0.400000,0.153846,0.222222
76,0.81,0.400000,0.153846,0.222222
77,0.82,0.400000,0.153846,0.222222
78,0.83,0.400000,0.153846,0.222222


In [19]:
# ============================================================
# STEP 223B — SELECT ENCRYPTED THRESHOLD
# ============================================================

eligible = encrypted_threshold_table[
    encrypted_threshold_table["recall"] >= 0.60
]

if len(eligible) > 0:

    best_encrypted = (
        eligible
        .sort_values(
            "f1",
            ascending=False
        )
        .iloc[0]
    )

else:

    best_encrypted = (
        encrypted_threshold_table
        .sort_values(
            "f1",
            ascending=False
        )
        .iloc[0]
    )

ENCRYPTED_THRESHOLD = float(
    best_encrypted["threshold"]
)

print(
    "Selected encrypted threshold:",
    ENCRYPTED_THRESHOLD
)

print(
    "Validation precision:",
    best_encrypted["precision"]
)

print(
    "Validation recall:",
    best_encrypted["recall"]
)

print(
    "Validation F1:",
    best_encrypted["f1"]
)

Selected encrypted threshold: 0.95
Validation precision: 0.5
Validation recall: 0.15384615384615385
Validation F1: 0.23529411764705882


In [20]:
# ============================================================
# STEP 224B — FINAL ENCRYPTED TEST
# ============================================================

enc_test_proba = (
    encrypted_hgb
    .predict_proba(X_enc_test)[:, 1]
)

enc_test_pred = (
    enc_test_proba >= ENCRYPTED_THRESHOLD
).astype(int)

enc_accuracy = accuracy_score(
    y_enc_test,
    enc_test_pred
)

enc_precision = precision_score(
    y_enc_test,
    enc_test_pred,
    zero_division=0
)

enc_recall = recall_score(
    y_enc_test,
    enc_test_pred,
    zero_division=0
)

enc_f1 = f1_score(
    y_enc_test,
    enc_test_pred,
    zero_division=0
)

print("==========================================")
print("FINAL ENCRYPTED TRAFFIC TEST")
print("==========================================")

print(
    "Threshold:",
    ENCRYPTED_THRESHOLD
)

print(
    "Accuracy:",
    f"{enc_accuracy:.4f}"
)

print(
    "Precision:",
    f"{enc_precision:.4f}"
)

print(
    "Recall:",
    f"{enc_recall:.4f}"
)

print(
    "F1:",
    f"{enc_f1:.4f}"
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_enc_test,
        enc_test_pred
    )
)

print("\nActual encrypted threats:")
print(
    int(y_enc_test.sum())
)

print("Detected encrypted threats:")
print(
    int(
        (
            (y_enc_test == 1)
            &
            (enc_test_pred == 1)
        ).sum()
    )
)

print("Missed encrypted threats:")
print(
    int(
        (
            (y_enc_test == 1)
            &
            (enc_test_pred == 0)
        ).sum()
    )
)

FINAL ENCRYPTED TRAFFIC TEST
Threshold: 0.95
Accuracy: 0.9993
Precision: 1.0000
Recall: 0.5652
F1: 0.7222

Confusion Matrix:
[[14229     0]
 [   10    13]]

Actual encrypted threats:
23
Detected encrypted threats:
13
Missed encrypted threats:
10


In [21]:
# ============================================================
# STEP 225B — ENCRYPTED FEATURE IMPORTANCE
# ============================================================

from sklearn.inspection import permutation_importance

encrypted_perm = permutation_importance(
    encrypted_hgb,
    X_enc_val,
    y_enc_val,
    scoring="f1",
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

encrypted_importance = pd.DataFrame({
    "Feature": encrypted_features,
    "Importance": encrypted_perm.importances_mean,
    "Std": encrypted_perm.importances_std
})

encrypted_importance = (
    encrypted_importance
    .sort_values(
        "Importance",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    encrypted_importance
)

,Feature,Importance,Std
0,encrypted_mean_duration,0.165913,0.024000
1,bytes_per_flow,0.156626,0.008833
2,encrypted_total_packets,0.131050,0.011304
3,bytes_change,0.089901,0.031190
4,encrypted_total_bytes,0.025602,0.004915
5,flow_count_change,0.006891,0.007798
6,destination_change,0.001449,0.002899
7,encrypted_flow_count,0.000000,0.000000
8,encrypted_unique_destinations,0.000000,0.000000
9,encrypted_unique_ports,0.000000,0.000000


In [22]:
# ============================================================
# STEP 226 — SAVE ENCRYPTED MODEL
# ============================================================

import joblib
from pathlib import Path

MODEL_DIR = project_root / "models"
MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

encrypted_model_bundle = {
    "model": encrypted_hgb,
    "threshold": ENCRYPTED_THRESHOLD,
    "features": encrypted_features,
}

encrypted_model_path = (
    MODEL_DIR / "encrypted_hgb.joblib"
)

joblib.dump(
    encrypted_model_bundle,
    encrypted_model_path
)

print("✅ Encrypted model saved")
print("Path:", encrypted_model_path)
print("Threshold:", ENCRYPTED_THRESHOLD)
print("Features:", len(encrypted_features))

✅ Encrypted model saved
Path: e:\CODEZILLA-SIH26145\models\encrypted_hgb.joblib
Threshold: 0.95
Features: 11


In [23]:
# ============================================================
# STEP 227 — VERIFY ENCRYPTED MODEL ARTIFACT
# ============================================================

loaded_encrypted = joblib.load(
    encrypted_model_path
)

print(
    "Loaded threshold:",
    loaded_encrypted["threshold"]
)

print(
    "Feature count:",
    len(
        loaded_encrypted["features"]
    )
)

print(
    "Model type:",
    type(
        loaded_encrypted["model"]
    ).__name__
)

print("\nFeatures:")

for feature in loaded_encrypted["features"]:
    print("-", feature)

Loaded threshold: 0.95
Feature count: 11
Model type: HistGradientBoostingClassifier

Features:
- encrypted_flow_count
- encrypted_total_packets
- encrypted_total_bytes
- encrypted_unique_destinations
- encrypted_unique_ports
- encrypted_mean_duration
- bytes_per_flow
- packets_per_flow
- flow_count_change
- bytes_change
- destination_change


In [24]:
from pathlib import Path
import sys

current = Path.cwd()

project_root = None

for candidate in [current, *current.parents]:

    if (candidate / "src").is_dir():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(
        "Project root not found."
    )

if str(project_root) not in sys.path:
    sys.path.insert(
        0,
        str(project_root)
    )

print("Project root:", project_root)

Project root: e:\CODEZILLA-SIH26145


In [25]:
from src.detectors.encrypted_detector import (
    detect,
    THRESHOLD,
    FEATURES
)

print("Encrypted threshold:", THRESHOLD)
print("Feature count:", len(FEATURES))

Encrypted threshold: 0.95
Feature count: 11


In [26]:
encrypted_production_results = detect(
    encrypted_window[FEATURES],
    top_k=5
)

encrypted_production_df = pd.DataFrame(
    encrypted_production_results
)

print(
    "Production encrypted results:",
    len(
        encrypted_production_results
    )
)

print("\nPrediction distribution:")

print(
    encrypted_production_df[
        "prediction"
    ].value_counts()
)

display(
    encrypted_production_df.head(20)
)

Production encrypted results: 47507

Prediction distribution:
prediction
BENIGN              47452
ENCRYPTED_THREAT       55
Name: count, dtype: int64


,prediction,model_score,decision_threshold,threat_class,severity,supporting_features
0,BENIGN,0.0000,0.95,ENCRYPTED_TRAFFIC,LOW,"[{'feature': 'encrypted_total_bytes', 'feature..."
1,BENIGN,0.0000,0.95,ENCRYPTED_TRAFFIC,LOW,"[{'feature': 'encrypted_total_bytes', 'feature..."
2,BENIGN,0.0001,0.95,ENCRYPTED_TRAFFIC,LOW,"[{'feature': 'encrypted_total_bytes', 'feature..."
3,BENIGN,0.0002,0.95,ENCRYPTED_TRAFFIC,LOW,"[{'feature': 'encrypted_total_bytes', 'feature..."
4,BENIGN,0.0006,0.95,ENCRYPTED_TRAFFIC,LOW,"[{'feature': 'encrypted_total_bytes', 'feature..."
5,BENIGN,0.0006,0.95,ENCRYPTED_TRAFFIC,LOW,"[{'feature': 'encrypted_total_bytes', 'feature..."
6,BENIGN,0.0000,0.95,ENCRYPTED_TRAFFIC,LOW,"[{'feature': 'encrypted_total_bytes', 'feature..."
7,BENIGN,0.0000,0.95,ENCRYPTED_TRAFFIC,LOW,"[{'feature': 'encrypted_total_bytes', 'feature..."
8,BENIGN,0.0004,0.95,ENCRYPTED_TRAFFIC,LOW,"[{'feature': 'encrypted_total_bytes', 'feature..."
9,BENIGN,0.0006,0.95,ENCRYPTED_TRAFFIC,LOW,"[{'feature': 'encrypted_total_bytes', 'feature..."


In [29]:
from src.threat_fusion import fuse_results

test_results = [
    {
        "prediction": "BENIGN",
        "model_score": 0.10,
        "threat_class": "DNS",
        "severity": "LOW",
        "supporting_features": []
    },
    {
        "prediction": "C2",
        "model_score": 0.82,
        "threat_class": "C2",
        "severity": "HIGH",
        "supporting_features": [
            {
                "feature": "pair_repetition_ratio",
                "feature_value": 0.72
            }
        ]
    },
    {
        "prediction": "ENCRYPTED_THREAT",
        "model_score": 0.97,
        "threat_class": "ENCRYPTED_TRAFFIC",
        "severity": "HIGH",
        "supporting_features": [
            {
                "feature": "bytes_per_flow",
                "feature_value": 85562
            }
        ]
    }
]

fusion_result = fuse_results(
    test_results
)

print(fusion_result)

{'prediction': 'THREAT', 'severity': 'HIGH', 'score': 0.97, 'primary_threat': 'ENCRYPTED_TRAFFIC', 'threats': [{'threat_class': 'ENCRYPTED_TRAFFIC', 'prediction': 'ENCRYPTED_THREAT', 'score': 0.97, 'severity': 'HIGH', 'supporting_features': [{'feature': 'bytes_per_flow', 'feature_value': 85562}]}, {'threat_class': 'C2', 'prediction': 'C2', 'score': 0.82, 'severity': 'HIGH', 'supporting_features': [{'feature': 'pair_repetition_ratio', 'feature_value': 0.72}]}], 'evidence': [{'feature': 'bytes_per_flow', 'feature_value': 85562}, {'feature': 'pair_repetition_ratio', 'feature_value': 0.72}]}


In [30]:
from src.detection_engine import analyze

print("✅ Detection engine imported")

✅ Detection engine imported


In [31]:
from src.feature_router import (
    describe_features,
    DNS_FEATURES,
    ENCRYPTED_FEATURES,
)

print("✅ Feature router imported")

print("\nDNS required features:", len(DNS_FEATURES))
print(
    "Encrypted required features:",
    len(ENCRYPTED_FEATURES)
)

✅ Feature router imported

DNS required features: 14
Encrypted required features: 11


In [33]:
print("dns_work:", "dns_work" in globals())
print("dns_source_window:", "dns_source_window" in globals())
print("encrypted_window:", "encrypted_window" in globals())

dns_work: False
dns_source_window: False
encrypted_window: True


In [36]:
# ============================================================
# LOAD PROCESSED DNS FEATURES
# ============================================================

from pathlib import Path
import pandas as pd

dns_artifact_path = (
    project_root
    / "data"
    / "processed"
    / "dns_source_features.parquet"
)

dns_work = pd.read_parquet(
    dns_artifact_path
)

print("✅ DNS features loaded")
print("Shape:", dns_work.shape)

✅ DNS features loaded
Shape: (40953, 25)


In [37]:
from src.feature_router import route_dns

dns_routed = route_dns(
    dns_work
)

print("✅ DNS routing works")
print("Shape:", dns_routed.shape)
print("Feature count:", len(dns_routed.columns))

✅ DNS routing works
Shape: (40953, 14)
Feature count: 14


In [38]:
# ============================================================
# SAVE ENCRYPTED PROCESSED FEATURES
# ============================================================

processed_dir = (
    project_root
    / "data"
    / "processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

encrypted_artifact_path = (
    processed_dir
    / "encrypted_source_features.parquet"
)

encrypted_window.to_parquet(
    encrypted_artifact_path,
    index=False
)

print("✅ Encrypted features saved")
print("Path:", encrypted_artifact_path)
print("Shape:", encrypted_window.shape)

✅ Encrypted features saved
Path: e:\CODEZILLA-SIH26145\data\processed\encrypted_source_features.parquet
Shape: (47507, 16)


In [39]:
from src.feature_router import route_encrypted

encrypted_routed = route_encrypted(
    encrypted_window
)

print("✅ Encrypted routing works")
print(
    "Shape:",
    encrypted_routed.shape
)

✅ Encrypted routing works
Shape: (47507, 11)


In [1]:
from pathlib import Path
import sys

current = Path.cwd()

project_root = None

for candidate in [current, *current.parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break

if project_root is None:
    raise RuntimeError(
        "Could not find project root containing src/"
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Project root:", project_root)
print(
    "detection_engine exists:",
    (
        project_root
        / "src"
        / "detection_engine.py"
    ).exists()
)

Project root: e:\CODEZILLA-SIH26145
detection_engine exists: True


In [2]:
import src.detection_engine as detection_engine

print(
    "Module loaded from:"
)

print(
    detection_engine.__file__
)

print(
    "\nanalyze_events exists:",
    hasattr(
        detection_engine,
        "analyze_events"
    )
)

print(
    "\nanalyze exists:",
    hasattr(
        detection_engine,
        "analyze"
    )
)

Module loaded from:
e:\CODEZILLA-SIH26145\src\detection_engine.py

analyze_events exists: True

analyze exists: False


In [3]:
from src.detection_engine import analyze_events

print("✅ Unified event engine imported")

✅ Unified event engine imported


In [7]:
# ============================================================
# LOAD SAVED ENCRYPTED FEATURES
# ============================================================

from pathlib import Path
import pandas as pd

if "project_root" not in globals():

    current = Path.cwd()

    project_root = None

    for candidate in [current, *current.parents]:

        if (candidate / "src").is_dir():
            project_root = candidate
            break

    if project_root is None:
        raise RuntimeError(
            "Could not find project root."
        )

encrypted_artifact_path = (
    project_root
    / "data"
    / "processed"
    / "encrypted_source_features.parquet"
)

print("Encrypted artifact:")
print(encrypted_artifact_path)

print(
    "File exists:",
    encrypted_artifact_path.exists()
)

if not encrypted_artifact_path.exists():
    raise FileNotFoundError(
        f"Encrypted feature artifact not found:\n"
        f"{encrypted_artifact_path}"
    )

encrypted_window = pd.read_parquet(
    encrypted_artifact_path
)

print("\n✅ encrypted_window loaded")
print("Shape:", encrypted_window.shape)

print("\nColumns:")
print(list(encrypted_window.columns))

Encrypted artifact:
e:\CODEZILLA-SIH26145\data\processed\encrypted_source_features.parquet
File exists: True

✅ encrypted_window loaded
Shape: (47507, 16)

Columns:
['SrcAddr', 'time_window', 'encrypted_flow_count', 'encrypted_total_packets', 'encrypted_total_bytes', 'encrypted_unique_destinations', 'encrypted_unique_ports', 'encrypted_mean_duration', 'encrypted_target', 'bytes_per_flow', 'packets_per_flow', 'flow_count_prev', 'flow_count_change', 'bytes_prev', 'bytes_change', 'destination_change']


In [8]:
# ============================================================
# CHECK BOTH FEATURE DATASETS
# ============================================================

print(
    "dns_work:",
    "dns_work" in globals()
)

print(
    "encrypted_window:",
    "encrypted_window" in globals()
)

dns_work: True
encrypted_window: True


In [9]:
# ============================================================
# STEP 241 — REAL UNIFIED EVENT ANALYSIS
# ============================================================

from src.detection_engine import analyze_events

unified_events = analyze_events(
    dns_data=dns_work,
    encrypted_data=encrypted_window,
    top_k=5
)

print(
    "Unified events:",
    len(unified_events)
)

print("\nPrediction distribution:")

print(
    unified_events[
        "prediction"
    ].value_counts()
)

display(
    unified_events.head(20)
)

c:\Users\magad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\magad\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\magad\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreatePro

Unified events: 80012

Prediction distribution:
prediction
BENIGN    79368
THREAT      644
Name: count, dtype: int64


,SrcAddr,time_window,prediction,severity,score,primary_threat,detector_count,threats,evidence
0,147.32.84.134,2011-08-10 09:46:50,BENIGN,LOW,0.0,None,0,[],[]
1,147.32.84.138,2011-08-10 09:46:50,BENIGN,LOW,0.0,None,0,[],[]
2,147.32.84.16,2011-08-10 09:46:50,BENIGN,LOW,0.0,None,0,[],[]
3,147.32.84.19,2011-08-10 09:46:50,BENIGN,LOW,0.0,None,0,[],[]
4,147.32.84.212,2011-08-10 09:46:50,BENIGN,LOW,0.0,None,0,[],[]
5,147.32.84.25,2011-08-10 09:46:50,BENIGN,LOW,0.0,None,0,[],[]
6,147.32.84.59,2011-08-10 09:46:50,BENIGN,LOW,0.0,None,0,[],[]
7,147.32.84.94,2011-08-10 09:46:50,BENIGN,LOW,0.0,None,0,[],[]
8,147.32.85.25,2011-08-10 09:46:50,BENIGN,LOW,0.0,None,0,[],[]
9,147.32.85.30,2011-08-10 09:46:50,BENIGN,LOW,0.0,None,0,[],[]


In [10]:
# ============================================================
# STEP 242 — THREAT EVENTS
# ============================================================

threat_events = unified_events[
    unified_events["prediction"] == "THREAT"
].copy()

print(
    "Threat events:",
    len(threat_events)
)

display(
    threat_events[
        [
            "SrcAddr",
            "time_window",
            "prediction",
            "severity",
            "score",
            "primary_threat",
            "detector_count",
        ]
    ].head(30)
)

Threat events: 644


,SrcAddr,time_window,prediction,severity,score,primary_threat,detector_count
571,38.229.1.72,2011-08-10 09:49:10,THREAT,HIGH,0.9513,DNS,1
3579,147.32.86.194,2011-08-10 10:04:10,THREAT,MEDIUM,0.9348,DNS,1
6837,69.164.208.51,2011-08-10 10:19:50,THREAT,MEDIUM,0.9389,DNS,1
7721,147.32.87.49,2011-08-10 10:24:10,THREAT,HIGH,0.9934,DNS,1
9355,147.32.84.132,2011-08-10 10:31:50,THREAT,HIGH,0.9788,DNS,1
15565,147.32.86.192,2011-08-10 11:01:00,THREAT,MEDIUM,0.9430,DNS,1
16753,147.32.84.165,2011-08-10 11:07:00,THREAT,HIGH,0.9982,ENCRYPTED_TRAFFIC,1
16987,147.32.84.131,2011-08-10 11:08:10,THREAT,HIGH,0.9683,DNS,1
17135,147.32.84.165,2011-08-10 11:08:50,THREAT,HIGH,0.9994,ENCRYPTED_TRAFFIC,1
17414,147.32.84.165,2011-08-10 11:10:10,THREAT,HIGH,0.9994,ENCRYPTED_TRAFFIC,1


In [11]:
# ============================================================
# STEP 243 — MULTI-DETECTOR AGREEMENT
# ============================================================

multi_detector_events = unified_events[
    unified_events["detector_count"] >= 2
].copy()

print(
    "Multi-detector events:",
    len(multi_detector_events)
)

display(
    multi_detector_events[
        [
            "SrcAddr",
            "time_window",
            "severity",
            "score",
            "primary_threat",
            "detector_count",
            "threats",
        ]
    ].head(20)
)

Multi-detector events: 3


,SrcAddr,time_window,severity,score,primary_threat,detector_count,threats
23767,147.32.84.165,2011-08-10 11:41:10,HIGH,1.0,ENCRYPTED_TRAFFIC,2,"[{'threat_class': 'ENCRYPTED_TRAFFIC', 'predic..."
42387,147.32.84.165,2011-08-10 13:12:30,HIGH,1.0,ENCRYPTED_TRAFFIC,2,"[{'threat_class': 'ENCRYPTED_TRAFFIC', 'predic..."
43675,147.32.84.165,2011-08-10 13:18:20,HIGH,1.0,ENCRYPTED_TRAFFIC,2,"[{'threat_class': 'ENCRYPTED_TRAFFIC', 'predic..."


In [12]:
# ============================================================
# STEP 244 — CURRENT DETECTOR SUMMARY
# ============================================================

print("Threat events by primary threat:")
print(
    threat_events[
        "primary_threat"
    ].value_counts()
)

print("\nThreat events by severity:")
print(
    threat_events[
        "severity"
    ].value_counts()
)

print("\nMulti-detector events:")
print(
    multi_detector_events[
        [
            "SrcAddr",
            "time_window",
            "severity",
            "score",
            "primary_threat",
            "detector_count"
        ]
    ]
)

Threat events by primary threat:
primary_threat
DNS                  589
ENCRYPTED_TRAFFIC     55
Name: count, dtype: int64

Threat events by severity:
severity
HIGH      622
MEDIUM     22
Name: count, dtype: int64

Multi-detector events:
             SrcAddr         time_window severity  score     primary_threat  \
23767  147.32.84.165 2011-08-10 11:41:10     HIGH    1.0  ENCRYPTED_TRAFFIC   
42387  147.32.84.165 2011-08-10 13:12:30     HIGH    1.0  ENCRYPTED_TRAFFIC   
43675  147.32.84.165 2011-08-10 13:18:20     HIGH    1.0  ENCRYPTED_TRAFFIC   

       detector_count  
23767               2  
42387               2  
43675               2  


In [13]:
from src.detector_adapter import normalize_result

test_dns = {
    "prediction": "DNS_THREAT",
    "model_score": 0.93,
    "decision_threshold": 0.93,
    "threat_class": "DNS",
    "severity": "HIGH",
    "supporting_features": [
        {
            "feature": "dns_query_rate",
            "feature_value": 56.8
        }
    ]
}

normalized_dns = normalize_result(
    test_dns,
    "DNS"
)

print(normalized_dns)

{'detector': 'DNS', 'prediction': 'DNS_THREAT', 'model_score': 0.93, 'threat_class': 'DNS', 'severity': 'HIGH', 'supporting_features': [{'feature': 'dns_query_rate', 'feature_value': 56.8}]}


In [14]:
test_ddos = {
    "prediction": "ATTACK",
    "model_score": 0.9991,
    "threat_class": "DDoS",
    "supporting_features": [
        {
            "feature": "total_packets",
            "feature_value": 401246
        }
    ]
}

normalized_ddos = normalize_result(
    test_ddos,
    "DDoS"
)

print(normalized_ddos)

{'detector': 'DDoS', 'prediction': 'ATTACK', 'model_score': 0.9991, 'threat_class': 'DDoS', 'severity': 'HIGH', 'supporting_features': [{'feature': 'total_packets', 'feature_value': 401246}]}


In [15]:
test_c2 = {
    "prediction": "SUSPICIOUS",
    "model_score": 0.6129,
    "threat_class": "C2",
    "severity": "MEDIUM",
    "supporting_features": [
        {
            "signal": "destination_concentration",
            "value": 0.75,
            "contribution": 0.15
        }
    ]
}

normalized_c2 = normalize_result(
    test_c2,
    "C2"
)

print(normalized_c2)

{'detector': 'C2', 'prediction': 'SUSPICIOUS', 'model_score': 0.6129, 'threat_class': 'C2', 'severity': 'MEDIUM', 'supporting_features': [{'signal': 'destination_concentration', 'value': 0.75, 'contribution': 0.15}]}


In [16]:
import importlib
import src.detection_engine as detection_engine

detection_engine = importlib.reload(
    detection_engine
)

print("✅ Four-detector engine loaded")
print(
    "analyze_events:",
    hasattr(
        detection_engine,
        "analyze_events"
    )
)

✅ Four-detector engine loaded
analyze_events: True


In [17]:
# ============================================================
# STEP 251 — FIND EXISTING DETECTOR DATA
# ============================================================

possible_names = [
    "X_stream_val",
    "X_stream_test",
    "X_stream_train",

    "source_c2",
    "source_c2_v3",
    "c2_features",
    "c2_model_data",

    "ddos_features",
    "ddos_model_data",
    "stream_features",

    "dns_work",
    "encrypted_window",
]

for name in possible_names:

    if name in globals():

        value = globals()[name]

        try:
            print(
                f"{name}: {value.shape}"
            )
        except AttributeError:
            print(
                f"{name}: exists"
            )

dns_work: (40953, 25)
encrypted_window: (47507, 16)


In [18]:
# ============================================================
# GET A REAL ENCRYPTED-THREAT ROW FOR API TESTING
# ============================================================

import json
import pandas as pd

# Load the saved encrypted feature artifact
enc = pd.read_parquet(
    r"E:\CODEZILLA-SIH26145\data\processed\encrypted_source_features.parquet"
)

print("Shape:", enc.shape)
print("Columns:")
print(list(enc.columns))

# Find the target column
target_col = None

for candidate in [
    "encrypted_target",
    "target",
    "Label"
]:
    if candidate in enc.columns:
        target_col = candidate
        break

if target_col is None:
    raise RuntimeError(
        "Could not find encrypted target column."
    )

print("\nTarget column:", target_col)
print(
    enc[target_col].value_counts()
)

# ------------------------------------------------------------
# Select a REAL positive encrypted-threat row
# ------------------------------------------------------------

positive = enc[
    enc[target_col] == 1
].copy()

if positive.empty:
    raise RuntimeError(
        "No encrypted-threat rows found."
    )

row = positive.iloc[0]

# ------------------------------------------------------------
# Exact 11 model features
# ------------------------------------------------------------

ENCRYPTED_FEATURES = [
    "encrypted_flow_count",
    "encrypted_total_packets",
    "encrypted_total_bytes",
    "encrypted_unique_destinations",
    "encrypted_unique_ports",
    "encrypted_mean_duration",
    "bytes_per_flow",
    "packets_per_flow",
    "flow_count_change",
    "bytes_change",
    "destination_change",
]

missing = [
    f
    for f in ENCRYPTED_FEATURES
    if f not in enc.columns
]

if missing:
    raise RuntimeError(
        "Missing encrypted features: "
        + ", ".join(missing)
    )

payload = {
    "source": (
        str(row["SrcAddr"])
        if "SrcAddr" in enc.columns
        else None
    ),

    "time_window": (
        str(row["time_window"])
        if "time_window" in enc.columns
        else None
    ),

    "encrypted_features": {
        feature: float(row[feature])
        for feature in ENCRYPTED_FEATURES
    }
}

print("\n========================================")
print("REAL ENCRYPTED THREAT API PAYLOAD")
print("========================================")

print(
    json.dumps(
        payload,
        indent=2
    )
)

Shape: (47507, 16)
Columns:
['SrcAddr', 'time_window', 'encrypted_flow_count', 'encrypted_total_packets', 'encrypted_total_bytes', 'encrypted_unique_destinations', 'encrypted_unique_ports', 'encrypted_mean_duration', 'encrypted_target', 'bytes_per_flow', 'packets_per_flow', 'flow_count_prev', 'flow_count_change', 'bytes_prev', 'bytes_change', 'destination_change']

Target column: encrypted_target
encrypted_target
0    47433
1       74
Name: count, dtype: int64

REAL ENCRYPTED THREAT API PAYLOAD
{
  "source": "147.32.84.165",
  "time_window": "2011-08-10 11:07:00",
  "encrypted_features": {
    "encrypted_flow_count": 1.0,
    "encrypted_total_packets": 7.0,
    "encrypted_total_bytes": 558.0,
    "encrypted_unique_destinations": 1.0,
    "encrypted_unique_ports": 1.0,
    "encrypted_mean_duration": 9.560554,
    "bytes_per_flow": 558.0,
    "packets_per_flow": 7.0,
    "flow_count_change": 0.0,
    "bytes_change": 192.0,
    "destination_change": 0.0
  }
}
